# Baseline Forecasting

Compare direction-specific naive, seasonal-naive, clock-time moving-average, and linear-regression forecasts. All predictions are one-step-ahead rolling-origin: only earlier observed ridership is available to historical baselines. Validation selects the model; test metrics are final evaluation only. No advanced models are trained.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

project_root = Path.cwd().resolve()
if project_root.name == 'Notebooks':
    project_root = project_root.parent
data_dir = project_root / 'DataProcessed'
target = 'ridership_count'
split_files = {
    'train': data_dir / 'metro_weather_train_prepared.csv',
    'validation': data_dir / 'metro_weather_validation_prepared.csv',
    'test': data_dir / 'metro_weather_test_prepared.csv'
}
splits = {name: pd.read_csv(path, parse_dates=['time']) for name, path in split_files.items()}
for name, frame in splits.items():
    assert target in frame.columns
    assert not frame.duplicated(['time', 'direction']).any()
    assert frame[['time', 'direction', target]].notna().all().all()
    assert set(frame['direction']) == {'Entry', 'Exit'}
assert splits['train']['time'].max() < splits['validation']['time'].min()
assert splits['validation']['time'].max() < splits['test']['time'].min()
feature_columns = [column for column in splits['train'].columns if column not in {'time', target}]
assert 'direction' in feature_columns and target not in feature_columns and 'time' not in feature_columns
print('Rows by split:', {name: len(frame) for name, frame in splits.items()})
print('Stage 4 regression predictors:', len(feature_columns))

Rows by split: {'train': 3780, 'validation': 810, 'test': 810}
Stage 4 regression predictors: 30


In [10]:
# Build each historical forecast from the complete observed ridership timeline, separately by direction.
history_path = data_dir / 'metro_weather_features_final.csv'
history_rows = pd.read_csv(history_path, parse_dates=['time']).sort_values(['time', 'direction']).reset_index(drop=True)
assert not history_rows.duplicated(['time', 'direction']).any()
assert history_rows[['time', 'direction', target]].notna().all().all()
history_rows['naive_previous_observation'] = history_rows.groupby('direction')[target].shift(1)
seasonal_lookup = history_rows[['time', 'direction', target]].copy()
seasonal_lookup['time'] += pd.Timedelta(hours=24)
seasonal_lookup = seasonal_lookup.rename(columns={target: 'seasonal_naive_24h'})
history_rows = history_rows.merge(seasonal_lookup, on=['time', 'direction'], how='left', validate='one_to_one')
for window in [3, 6, 24]:
    prediction_column = f'moving_average_{window}h'
    history_rows[prediction_column] = np.nan
    for direction, group in history_rows.groupby('direction', sort=False):
        ordered = group.sort_values('time')
        historical_mean = ordered.set_index('time')[target].rolling(
            f'{window}h', closed='left', min_periods=1
        ).mean()
        history_rows.loc[ordered.index, prediction_column] = historical_mean.to_numpy()

# Keep forecast rows only where Stage 4 retained a valid feature/target sample.
evaluation_keys = pd.concat([
    frame[['time', 'direction']].assign(split=name)
    for name, frame in splits.items()
], ignore_index=True)
all_rows = history_rows.merge(evaluation_keys, on=['time', 'direction'], how='inner', validate='one_to_one')
assert len(all_rows) == sum(len(frame) for frame in splits.values())
assert not all_rows.duplicated(['time', 'direction']).any()
baseline_models = [
    'naive_previous_observation', 'seasonal_naive_24h',
    'moving_average_3h', 'moving_average_6h', 'moving_average_24h'
]
print('Historical forecasts use the complete ridership timeline and prior timestamps only.')

Historical forecasts use the complete ridership timeline and prior timestamps only.


In [11]:
def encode_features(frame, columns=None):
    encoded = pd.get_dummies(frame[feature_columns], columns=['direction'], dtype=float)
    return encoded if columns is None else encoded.reindex(columns=columns, fill_value=0)

def forecast_metrics(actual, predicted):
    valid = predicted.notna() & actual.notna()
    observed = actual.loc[valid].astype(float)
    forecast = predicted.loc[valid].astype(float)
    errors = forecast - observed
    nonzero = observed.ne(0)
    mape = (np.abs(errors.loc[nonzero] / observed.loc[nonzero]).mean() * 100) if nonzero.any() else np.nan
    return {'n_predictions': int(valid.sum()), 'n_nonzero_for_mape': int(nonzero.sum()),
            'MAE': float(np.abs(errors).mean()), 'RMSE': float(np.sqrt(np.mean(errors ** 2))), 'MAPE_pct': float(mape)}

train_rows = splits['train'].sort_values(['time', 'direction']).reset_index(drop=True)
validation_rows = splits['validation'].sort_values(['time', 'direction']).reset_index(drop=True)
train_X = encode_features(train_rows)
validation_X = encode_features(validation_rows, train_X.columns)
validation_model = LinearRegression().fit(train_X, train_rows[target])
validation_predictions = pd.Series(validation_model.predict(validation_X))

validation_results = all_rows.loc[all_rows['split'].eq('validation'), ['time', 'direction', target]].reset_index(drop=True).copy()
for model in baseline_models:
    validation_results[model] = all_rows.loc[all_rows['split'].eq('validation'), model].to_numpy()
validation_results['linear_regression'] = validation_predictions.to_numpy()
comparison_models = baseline_models + ['linear_regression']
validation_eligible = validation_results[comparison_models].notna().all(axis=1)
validation_results = validation_results.loc[validation_eligible].reset_index(drop=True)
assert validation_results.groupby('direction').size().nunique() == 1
validation_metrics = pd.DataFrame([
    {'model': model, **forecast_metrics(validation_results[target], validation_results[model])}
    for model in comparison_models
]).sort_values(['MAE', 'RMSE']).reset_index(drop=True)
assert validation_metrics['n_predictions'].nunique() == 1
selected_model = validation_metrics.iloc[0]['model']
print('Validation metrics on identical eligible timestamp-direction keys (selection by MAE):')
display(validation_metrics)
print('Shared validation keys:', len(validation_results))
print('Selected using validation only:', selected_model)

Validation metrics on identical eligible timestamp-direction keys (selection by MAE):


,model,n_predictions,n_nonzero_for_mape,MAE,RMSE,MAPE_pct
0,linear_regression,810,810,15309.669837,19916.905574,872.771082
1,seasonal_naive_24h,810,810,15899.324691,33482.029178,10.252179
2,naive_previous_observation,810,810,40960.871605,53228.127751,399.904562
3,moving_average_3h,810,810,65043.019753,83256.250933,2487.640211
4,moving_average_24h,810,810,71615.539271,89310.307188,8297.704539
5,moving_average_6h,810,810,80699.131276,101104.920217,8093.078149


Shared validation keys: 810
Selected using validation only: linear_regression


In [12]:
# Refit regression only after validation selection, using train+validation. Test targets are not used for fitting or selection.
train_validation = pd.concat([splits['train'], splits['validation']], ignore_index=True).sort_values(['time', 'direction']).reset_index(drop=True)
train_validation_X = encode_features(train_validation)
test_rows = splits['test'].sort_values(['time', 'direction']).reset_index(drop=True)
test_X = encode_features(test_rows, train_validation_X.columns)
final_regression = LinearRegression().fit(train_validation_X, train_validation[target])
test_regression_predictions = final_regression.predict(test_X)

test_results = all_rows.loc[all_rows['split'].eq('test'), ['time', 'direction', target]].reset_index(drop=True).copy()
for model in baseline_models:
    test_results[model] = all_rows.loc[all_rows['split'].eq('test'), model].to_numpy()
test_results['linear_regression'] = test_regression_predictions
assert test_rows[['time', 'direction']].equals(test_results[['time', 'direction']])
test_eligible = test_results[comparison_models].notna().all(axis=1)
test_results = test_results.loc[test_eligible].reset_index(drop=True)
assert test_results.groupby('direction').size().nunique() == 1
test_metrics = pd.DataFrame([
    {'model': model, **forecast_metrics(test_results[target], test_results[model])}
    for model in comparison_models
]).sort_values(['MAE', 'RMSE']).reset_index(drop=True)
assert test_metrics['n_predictions'].nunique() == 1

metrics = pd.concat([validation_metrics.assign(split='validation'), test_metrics.assign(split='test')], ignore_index=True)
predictions = []
for split_name, results in [('validation', validation_results), ('test', test_results)]:
    for model in comparison_models:
        output = results[['time', 'direction', target, model]].rename(columns={target: 'actual', model: 'prediction'})
        assert output['prediction'].notna().all()
        output['split'] = split_name
        output['model'] = model
        predictions.append(output[['split', 'model', 'time', 'direction', 'actual', 'prediction']])
predictions = pd.concat(predictions, ignore_index=True)
assert not predictions.duplicated(['split', 'model', 'time', 'direction']).any()
assert predictions.groupby(['split', 'model']).size().nunique() == 1

metrics_path = data_dir / 'baseline_forecasting_metrics.csv'
predictions_path = data_dir / 'baseline_forecasting_predictions.csv'
metrics.to_csv(metrics_path, index=False)
predictions.to_csv(predictions_path, index=False)
print('Test metrics on identical eligible timestamp-direction keys (final evaluation only):')
display(test_metrics)
print('Shared eligible keys:', {'validation': len(validation_results), 'test': len(test_results)})
print('MAPE excludes zero actuals; nonzero actual count:', int(test_metrics['n_nonzero_for_mape'].min()))
print('Saved:', metrics_path, 'and', predictions_path)

Test metrics on identical eligible timestamp-direction keys (final evaluation only):


,model,n_predictions,n_nonzero_for_mape,MAE,RMSE,MAPE_pct
0,linear_regression,810,810,15808.969128,21902.348972,923.808366
1,seasonal_naive_24h,810,810,22585.714815,42670.234333,16.299590
2,naive_previous_observation,810,810,37784.359259,50584.197426,383.445265
3,moving_average_3h,810,810,62150.890123,80152.984101,2199.608789
4,moving_average_24h,810,810,67061.801764,85512.693725,6617.872430
5,moving_average_6h,810,810,77815.159877,97245.458703,6542.377210


Shared eligible keys: {'validation': 810, 'test': 810}
MAPE excludes zero actuals; nonzero actual count: 810
Saved: /Users/shivaya/Desktop/Desktop - Utkarsh’s MacBook Pro/Research/DelhiMetroResearch/DataProcessed/baseline_forecasting_metrics.csv and /Users/shivaya/Desktop/Desktop - Utkarsh’s MacBook Pro/Research/DelhiMetroResearch/DataProcessed/baseline_forecasting_predictions.csv


## Key Observations

The following output summarizes validation selection, final test comparisons, and implications for the next modelling stage.

In [13]:
best_baseline = validation_metrics.loc[validation_metrics['model'].isin(baseline_models)].iloc[0]
selected_validation = validation_metrics.loc[validation_metrics['model'].eq(selected_model)].iloc[0]
selected_test = test_metrics.loc[test_metrics['model'].eq(selected_model)].iloc[0]
best_test = test_metrics.iloc[0]
print(f"Best baseline on validation: {best_baseline['model']} (MAE {best_baseline['MAE']:.2f}, RMSE {best_baseline['RMSE']:.2f}, MAPE {best_baseline['MAPE_pct']:.2f}%).")
print(f"Selected by validation: {selected_model} (MAE {selected_validation['MAE']:.2f}, RMSE {selected_validation['RMSE']:.2f}, MAPE {selected_validation['MAPE_pct']:.2f}%).")
print(f"Selected model final test: MAE {selected_test['MAE']:.2f}, RMSE {selected_test['RMSE']:.2f}, MAPE {selected_test['MAPE_pct']:.2f}%; best test comparison was {best_test['model']} (MAE {best_test['MAE']:.2f}).")
print('Next stage: use the chronological baseline as a benchmark, then test carefully tuned forecasting models and assess errors by direction and time period.')

Best baseline on validation: seasonal_naive_24h (MAE 15899.32, RMSE 33482.03, MAPE 10.25%).
Selected by validation: linear_regression (MAE 15309.67, RMSE 19916.91, MAPE 872.77%).
Selected model final test: MAE 15808.97, RMSE 21902.35, MAPE 923.81%; best test comparison was linear_regression (MAE 15808.97).
Next stage: use the chronological baseline as a benchmark, then test carefully tuned forecasting models and assess errors by direction and time period.


## Audit Findings

- **Issues and fixes:** Prior scores used unequal samples, and historical baselines omitted ridership rows excluded by Stage 4. Recomputed forecasts from the full observed ridership history and evaluated all models on the same keys.
- **Validation selection:** Linear regression; MAE 15,309.67, RMSE 19,916.91, MAPE 872.77%.
- **Test comparison:** Linear regression MAE 15,808.97, RMSE 21,902.35, MAPE 923.81%; seasonal naive MAE 22,585.71, RMSE 42,670.23, MAPE 16.30%.
- **Integrity:** Each model uses 810 aligned timestamp-direction cases per split. Historical forecasts use past observations only; seasonal naive uses exact 24-hour matches. Regression fitting/encoding is training-only for validation, then refit on train+validation after validation selection. Test data was not used for selection.
- **Limitations:** MAPE excludes zero actuals but is inflated by small positive ridership values. Prediction-hour weather is assumed available. These baselines are suitable benchmarks for Stage 6, not deployment-ready forecasts.

In [14]:
audit_metrics = pd.read_csv(metrics_path)
audit_predictions = pd.read_csv(predictions_path, parse_dates=['time'])
for split_name, source_frame in [('validation', splits['validation']), ('test', splits['test'])]:
    split_predictions = audit_predictions.loc[audit_predictions['split'].eq(split_name)]
    source_keys = set(map(tuple, source_frame[['time', 'direction']].to_numpy()))
    model_keys = [set(map(tuple, group[['time', 'direction']].to_numpy())) for _, group in split_predictions.groupby('model')]
    assert model_keys and all(keys == model_keys[0] for keys in model_keys)
    assert model_keys[0].issubset(source_keys)
    assert split_predictions['prediction'].notna().all()
    assert not split_predictions.duplicated(['model', 'time', 'direction']).any()
    for model, group in split_predictions.groupby('model'):
        errors = group['prediction'].to_numpy() - group['actual'].to_numpy()
        nonzero_actual = group['actual'].ne(0).to_numpy()
        recalculated = {
            'MAE': np.abs(errors).mean(),
            'RMSE': np.sqrt(np.mean(errors ** 2)),
            'MAPE_pct': np.abs(errors[nonzero_actual] / group['actual'].to_numpy()[nonzero_actual]).mean() * 100
        }
        stored = audit_metrics.loc[audit_metrics['split'].eq(split_name) & audit_metrics['model'].eq(model)].iloc[0]
        assert all(np.isclose(recalculated[key], stored[key]) for key in recalculated)
    print(f'{split_name}: {len(model_keys[0])} shared aligned keys; metrics independently reproduced.')

print('Validation metrics:')
display(audit_metrics.loc[audit_metrics['split'].eq('validation')].sort_values('MAE')[['model', 'n_predictions', 'MAE', 'RMSE', 'MAPE_pct']])
print('Test metrics (evaluation only):')
display(audit_metrics.loc[audit_metrics['split'].eq('test')].sort_values('MAE')[['model', 'n_predictions', 'MAE', 'RMSE', 'MAPE_pct']])
actuals = audit_predictions.loc[audit_predictions['split'].eq('test') & audit_predictions['model'].eq(selected_model), 'actual']
print(f"MAPE explanation: scored actuals are all nonzero; minimum={actuals.min():.0f}, 1st percentile={actuals.quantile(0.01):.0f}. Small positive denominators magnify percentage error.")
print(f'Validation-selected model: {selected_model}; test metrics did not affect selection.')

validation: 810 shared aligned keys; metrics independently reproduced.
test: 810 shared aligned keys; metrics independently reproduced.
Validation metrics:


,model,n_predictions,MAE,RMSE,MAPE_pct
0,linear_regression,810,15309.669837,19916.905574,872.771082
1,seasonal_naive_24h,810,15899.324691,33482.029178,10.252179
2,naive_previous_observation,810,40960.871605,53228.127751,399.904562
3,moving_average_3h,810,65043.019753,83256.250933,2487.640211
4,moving_average_24h,810,71615.539271,89310.307188,8297.704539
5,moving_average_6h,810,80699.131276,101104.920217,8093.078149


Test metrics (evaluation only):


,model,n_predictions,MAE,RMSE,MAPE_pct
6,linear_regression,810,15808.969128,21902.348972,923.808366
7,seasonal_naive_24h,810,22585.714815,42670.234333,16.299590
8,naive_previous_observation,810,37784.359259,50584.197426,383.445265
9,moving_average_3h,810,62150.890123,80152.984101,2199.608789
10,moving_average_24h,810,67061.801764,85512.693725,6617.872430
11,moving_average_6h,810,77815.159877,97245.458703,6542.377210


MAPE explanation: scored actuals are all nonzero; minimum=29, 1st percentile=63. Small positive denominators magnify percentage error.
Validation-selected model: linear_regression; test metrics did not affect selection.
